# Retrieval Head Ablation (Step 2)

### Validate identified retrieval heads via targeted ablation on held-out data

In [1]:
%pip install torch transformers numpy huggingface_hub accelerate bitsandbytes python-dotenv pandas scikit-learn quantulum3

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 23.4 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=c9545b42b2bdf18b6440ff919a4addbb81b27430d5df462fc9e431c17680cc64
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [2]:
import os
import re
import json
import random
import torch
import numpy as np
import pandas as pd
from contextlib import nullcontext
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from dotenv import load_dotenv
from quantulum3 import parser

#### If using Google Colab: Run The Following Cells (Ignore if using some other environment)

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

In [4]:
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF_TOKEN loaded and Hugging Face login successful.")
else:
    print("HF_TOKEN not found.")

HF_TOKEN loaded and Hugging Face login successful.


#### If using a high end GPU not from Colab, but from Lambda Labs or Colab GPUs but locally in VSCode

In [ ]:
load_dotenv(override=False)

HF_TOKEN = os.getenv("HF_TOKEN") or globals().get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("HF_TOKEN loaded and Hugging Face login successful.")
else:
    print("HF_TOKEN not found. Add HF_TOKEN=... to your .env file or set it in the environment.")

#### Configuration & Hyperparameters

In [10]:
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

TARGET_SEQ_LEN = 7000
NEEDLE_DEPTH = 0.5
SPLIT = 0.8
SPLIT_SEED = 42

TASKS = [
    {"id": "registrant_name", "question": "What is the registrant's name?"},
    {"id": "headquarters_city", "question": "What is the registrant's headquarters city?"},
    {"id": "headquarters_state", "question": "What is the registrant's headquarters state?"},
    {"id": "incorporation_state", "question": "What is the registrant's incorporation state?"},
    {"id": "incorporation_year", "question": "What is the incorporation year?"},
    {"id": "employees_count_total", "question": "How many total employees does the registrant have?"},
    {"id": "holder_record_amount", "question": "What is the number of holders of record of the registrant's common stock?"},
    {"id": "employees_count_full_time", "question": "How many full-time employees does the registrant have?"},
    {"id": "ceo_lastname", "question": "What is the CEO's last name?"},
]
TASK_MAP = {t["id"]: t for t in TASKS}

# Paths
DATA_PATH = "/content/cleaned_EDGAR_gt_2-22-2026.csv"

RANKING_RUN_NAME = "rank_2026-03-22_15-17-03" # Update this to your ranking run
HEADS_JSON_PATH = f"/content/{RANKING_RUN_NAME}/ranked_heads.json"

import datetime
TIMESTAMP = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
ABLATION_RUN_NAME = f"ablation_{TIMESTAMP}"
ABLATION_OUTPUT_DIR = f"/content/{ABLATION_RUN_NAME}"

# Ablation-specific config
ABLATION_K_VALUES = [8, 16, 32, 48, 64, 96, 128]
MAX_DECODE_TOKENS = 100

# Error logging config
ENABLE_ERROR_LOGGING = True
EXPERIMENT_NAME = "retrieval_head_ablation"
ERROR_LOG_DIR = os.path.join(ABLATION_OUTPUT_DIR, "error_logs")
WRONG_PREDICTIONS_JSON = os.path.join(ERROR_LOG_DIR, "wrong_predictions.json")
WRONG_PREDICTIONS_CSV = os.path.join(ERROR_LOG_DIR, "wrong_predictions.csv")
WRONG_PREDICTIONS_SUMMARY_JSON = os.path.join(ERROR_LOG_DIR, "wrong_predictions_summary.json")

# Model loading
TORCH_DTYPE = torch.bfloat16
ATTN_IMPL = "eager"

print("Configuration loaded.")
print(f"Model: {MODEL_ID}")
print(f"Ablation run: {ABLATION_RUN_NAME}")
print(f"Ablation K values: {ABLATION_K_VALUES}")
print(f"Max decode tokens: {MAX_DECODE_TOKENS}")
print(f"Heads JSON: {HEADS_JSON_PATH}")
print(f"Error logging: {ENABLE_ERROR_LOGGING}")

Configuration loaded.
Model: meta-llama/Meta-Llama-3-8B-Instruct
Ablation run: ablation_2026-03-22_20-46-39
Ablation K values: [8, 16, 32, 48, 64, 96, 128]
Max decode tokens: 100
Heads JSON: /content/rank_2026-03-22_15-17-03/ranked_heads.json
Error logging: True


In [6]:
# If the right meta.json attributes are present, feel free to run this validation code to make sure this experiment will be consistent with the data you have
parent_directory = os.path.dirname(HEADS_JSON_PATH)
with open(os.path.join(parent_directory, "meta.json"), "r") as f:
    rank_heads_meta = json.load(f)
SOURCE_MODEL = rank_heads_meta.get("source_model")
TOP_K_HEADS = rank_heads_meta.get("top_k_heads")

if SOURCE_MODEL != MODEL_ID:
    print("WARNING:")
    print(f"SOURCE_MODEL from this data and current MODEL_ID differ\nSOURCE_MODEL: {SOURCE_MODEL} != MODEL_ID: {MODEL_ID}")

if ABLATION_K_VALUES[-1] != TOP_K_HEADS:
    print("WARNING:")
    print(f"Top K Heads from this data and current Max Number Of Ablations differ\nTop K Heads: {TOP_K_HEADS} != Max Number Of Ablations: {ABLATION_K_VALUES[-1]}")

##### Tokenizer & Model Loading

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: {MODEL_ID}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Tokenizer loaded: meta-llama/Meta-Llama-3-8B-Instruct
Vocab size: 128,000


In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
    attn_implementation=ATTN_IMPL,
)
model.eval()

print(f"Model loaded: {MODEL_ID}")
print(f"dtype: {next(model.parameters()).dtype}")
print(f"Num layers: {model.config.num_hidden_layers}")
print(f"Num heads (Q): {model.config.num_attention_heads}")
print(f"Num KV heads: {model.config.num_key_value_heads}")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded: meta-llama/Meta-Llama-3-8B-Instruct
dtype: torch.bfloat16
Num layers: 32
Num heads (Q): 32
Num KV heads: 8


---
## Section 2 -- Load Ablation Split & Identified Heads

In [11]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH)
df = df[df["haystack_token_length"] == TARGET_SEQ_LEN].reset_index(drop=True)

_, ablation_df = train_test_split(
    df,
    test_size=1 - SPLIT,
    stratify=df["task"],
    random_state=SPLIT_SEED,
)
ablation_df = ablation_df.reset_index(drop=True)

print(f"Ablation set: {len(ablation_df):,} samples")
print(ablation_df.groupby("task").size().to_string())

Ablation set: 265 samples
task
ceo_lastname                 34
employees_count_full_time    10
employees_count_total        29
headquarters_city            29
headquarters_state           28
holder_record_amount         29
incorporation_state          32
incorporation_year           32
registrant_name              42


In [12]:
with open(HEADS_JSON_PATH, "r") as f:
    heads_data = json.load(f)

# Per-task top-K heads as (layer, head) tuples
task_head_rankings = {}
for task_id, head_list in heads_data["tasks"].items():
    task_head_rankings[task_id] = [(h["layer"], h["head"]) for h in head_list]

# Shared heads
shared_heads = [(h["layer"], h["head"]) for h in heads_data["shared_heads"]]
global_heads = [(h["layer"], h["head"]) for h in heads_data["global_heads"]]

print(f"Loaded {len(task_head_rankings)} task rankings from {HEADS_JSON_PATH}")
for tid, heads in task_head_rankings.items():
    print(f"  {tid:<35} {len(heads)} heads")
print(f"Shared heads: {len(shared_heads)}")

Loaded 9 task rankings from /content/rank_2026-03-22_15-17-03/ranked_heads.json
  registrant_name                     128 heads
  headquarters_city                   128 heads
  headquarters_state                  128 heads
  incorporation_state                 128 heads
  incorporation_year                  128 heads
  employees_count_total               128 heads
  holder_record_amount                128 heads
  employees_count_full_time           128 heads
  ceo_lastname                        128 heads
Shared heads: 102


---
## Section 3 -- Core Ablation Infrastructure

In [13]:
class HeadAblationHooks:
    """Context manager that zeros out selected attention heads before o_proj."""

    def __init__(self, model, heads: list[tuple[int, int]]):
        self.model = model
        self.heads = heads
        self.handles = []
        self.head_dim = model.config.hidden_size // model.config.num_attention_heads

        # Group heads by layer for efficient hooking
        self.by_layer = {}
        for layer_idx, head_idx in heads:
            self.by_layer.setdefault(layer_idx, set()).add(head_idx)

    def __enter__(self):
        for layer_idx, layer_heads in self.by_layer.items():
            if layer_idx >= len(self.model.model.layers):
                continue
            o_proj = self.model.model.layers[layer_idx].self_attn.o_proj

            def pre_hook(_module, inputs, heads=sorted(layer_heads)):
                x = inputs[0]
                if x.ndim != 3:
                    return inputs
                x = x.clone()
                for head_idx in heads:
                    start = head_idx * self.head_dim
                    end = start + self.head_dim
                    x[..., start:end] = 0
                return (x,)

            self.handles.append(o_proj.register_forward_pre_hook(pre_hook))
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        for handle in self.handles:
            handle.remove()
        self.handles = []

In [14]:
def greedy_decode(model, tokenizer, prompt_ids, max_tokens=MAX_DECODE_TOKENS) -> str:
    """Prefill + token-by-token greedy decode. No attention output needed."""
    with torch.inference_mode():
        out = model(
            input_ids=prompt_ids[:, :-1],
            use_cache=True,
            return_dict=True,
            output_attentions=False,
        )
        past_kv = out.past_key_values
        inp = prompt_ids[:, -1:]
        position = prompt_ids.size(1) - 1
        device = prompt_ids.device

        generated = []
        for _ in range(max_tokens):
            pos_ids = torch.tensor([[position]], dtype=torch.long, device=device)
            out = model(
                input_ids=inp,
                past_key_values=past_kv,
                position_ids=pos_ids,
                use_cache=True,
                output_attentions=False,
                return_dict=True,
            )
            past_kv = out.past_key_values
            next_id = out.logits[:, -1].argmax(dim=-1)
            generated.append(next_id.item())
            if next_id.item() == tokenizer.eos_token_id:
                break
            position += 1
            inp = next_id.unsqueeze(1)

    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [15]:
def normalize_value(text: str) -> str:
    if not text or str(text).lower() == "nan":
        return ""

    s = str(text)

    try:
        quants = parser.parse(s)
        for q in quants:
            s = s.replace(q.surface, f"{q.value:g}")
    except Exception:
        pass

    s = re.sub(r"\s+", " ", s.replace("\n", " ")).strip().lower()
    s = s.replace(",", "")

    number_words = {
        "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
        "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9",
        "ten": "10", "eleven": "11", "twelve": "12", "thirteen": "13",
        "fourteen": "14", "fifteen": "15", "sixteen": "16", "seventeen": "17",
        "eighteen": "18", "nineteen": "19", "twenty": "20", "thirty": "30",
        "forty": "40", "fifty": "50", "sixty": "60", "seventy": "70",
        "eighty": "80", "ninety": "90", "hundred": "100", "thousand": "1000",
        "million": "1000000", "billion": "1000000000"
    }
    for word, digit in number_words.items():
        s = re.sub(r"\b" + word + r"\b", digit, s)
    return s

---
## Section 4 -- Prompt Building & Evaluation Helpers

In [16]:
CONTROL_TOKENS = ["<|begin_of_text|>", "<|end_of_text|>", "<|eot_id|>", "<|start_header_id|>", "<|end_header_id|>"]
TOKEN_PATTERN = re.compile("|".join(re.escape(t) for t in CONTROL_TOKENS))


def find_needle_span(
    prompt_ids: list[int],
    needle_ids: list[int],
    threshold: float = 0.9,
) -> tuple[int, int]:
    """Locate needle tokens inside the full tokenized prompt via sliding window overlap."""
    span_len = len(needle_ids)
    if span_len == 0:
        return -1, -1

    needle_set = set(needle_ids)

    for i in range(len(prompt_ids) - span_len + 1):
        window = set(prompt_ids[i : i + span_len])
        if len(window & needle_set) / len(needle_set) >= threshold:
            return i, i + span_len

    return -1, -1


def build_prompt(row: pd.Series, task: dict, tokenizer) -> dict:
    """Construct prompt and locate needle span in tokenized input."""
    haystack = TOKEN_PATTERN.sub("", row["haystack_text"]).strip()
    needle = row["needle_sentence"]


    mid = len(haystack) // 2
    context = haystack[:mid] + " " + needle + " " + haystack[mid:]

    message = f"<document>{context}</document>\nQuestion: {task['question']}\nAnswer:"
    messages = [{"role": "user", "content": message}]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"]

    # Locate needle span in the tokenized prompt
    needle_ids = tokenizer.encode(needle, add_special_tokens=False)
    needle_start, needle_end = find_needle_span(input_ids[0].tolist(), needle_ids)

    return {
        "input_ids": input_ids,
        "needle_start": needle_start,
        "needle_end": needle_end,
    }

In [17]:
def evaluate_sample(model, tokenizer, row, task, ablation_heads=None) -> dict:
    """Build prompt, optionally ablate heads, greedy decode, and compute normalized match."""
    prompt = build_prompt(row, task, tokenizer)

    if prompt["needle_start"] == -1:
        return {
            "decoded": "",
            "value_match": False,
            "skipped": True,
            "ground_truth": row.get("needle_value", ""),
            "normalized_ground_truth": "",
            "normalized_decoded": "",
            "needle_sentence": row.get("needle_sentence", ""),
        }

    input_ids = prompt["input_ids"].to(model.device)
    hook_ctx = HeadAblationHooks(model, ablation_heads) if ablation_heads else nullcontext()

    with hook_ctx:
        decoded = greedy_decode(model, tokenizer, input_ids)

    ground_truth = row.get("needle_value", "")
    normalized_ground_truth = normalize_value(ground_truth)
    normalized_decoded = normalize_value(decoded)
    value_match = normalized_ground_truth in normalized_decoded

    del input_ids
    torch.cuda.empty_cache()

    return {
        "decoded": decoded,
        "value_match": value_match,
        "skipped": False,
        "ground_truth": ground_truth,
        "normalized_ground_truth": normalized_ground_truth,
        "normalized_decoded": normalized_decoded,
        "needle_sentence": row.get("needle_sentence", ""),
    }

In [18]:
def collect_wrong_predictions(
    condition_results: dict,
    condition_name: str,
    experiment_type: str,
    heads_ablated: list[tuple[int, int]] | None = None,
) -> list[dict]:
    """Collect only mismatched rows from a condition evaluation result."""
    rows = []
    heads_ablated = heads_ablated or []

    for task_id, task_result in condition_results.items():
        samples = task_result.get("samples", [])
        for sample in samples:
            if sample.get("skipped", False) or sample.get("value_match", False):
                continue

            rows.append({
                "experiment_name": EXPERIMENT_NAME,
                "experiment_run_name": ABLATION_RUN_NAME,
                "experiment_type": experiment_type,
                "condition_name": condition_name,
                "task_id": task_id,
                "row_index": sample.get("row_index"),
                "filename": sample.get("filename"),
                "ground_truth_raw": sample.get("ground_truth", ""),
                "model_output_raw": sample.get("decoded", ""),
                "ground_truth_normalized": sample.get("normalized_ground_truth", ""),
                "model_output_normalized": sample.get("normalized_decoded", ""),
                "needle_sentence": sample.get("needle_sentence", ""),
                "value_match": bool(sample.get("value_match", False)),
                "heads_ablated_count": len(heads_ablated),
                "heads_ablated": [list(h) for h in heads_ablated],
            })

    return rows

In [19]:
def build_wrong_prediction_summary(records: list[dict]) -> dict:
    """Build compact summary stats for quick triage by condition and task."""
    summary = {
        "run_metadata": {
            "experiment_name": EXPERIMENT_NAME,
            "experiment_run_name": ABLATION_RUN_NAME,
            "source_ranking_run": RANKING_RUN_NAME,
            "timestamp": TIMESTAMP,
            "total_wrong_predictions": len(records),
        },
        "by_experiment_type": {},
        "by_condition": {},
        "by_task": {},
    }

    for rec in records:
        exp_type = rec["experiment_type"]
        cond = rec["condition_name"]
        task = rec["task_id"]

        summary["by_experiment_type"][exp_type] = summary["by_experiment_type"].get(exp_type, 0) + 1
        summary["by_condition"][cond] = summary["by_condition"].get(cond, 0) + 1
        summary["by_task"][task] = summary["by_task"].get(task, 0) + 1

    return summary

In [20]:
def write_wrong_prediction_logs(records: list[dict]) -> None:
    """Write wrong predictions to JSON + CSV + summary JSON."""
    os.makedirs(ERROR_LOG_DIR, exist_ok=True)

    payload = {
        "run_metadata": {
            "experiment_name": EXPERIMENT_NAME,
            "experiment_run_name": ABLATION_RUN_NAME,
            "source_ranking_run": RANKING_RUN_NAME,
            "timestamp": TIMESTAMP,
            "output_dir": ABLATION_OUTPUT_DIR,
        },
        "rows": records,
    }

    with open(WRONG_PREDICTIONS_JSON, "w") as f:
        json.dump(payload, f, indent=2)

    df_wrong = pd.DataFrame(records)
    df_wrong.to_csv(WRONG_PREDICTIONS_CSV, index=False)

    summary = build_wrong_prediction_summary(records)
    with open(WRONG_PREDICTIONS_SUMMARY_JSON, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved wrong prediction logs to: {ERROR_LOG_DIR}")
    print(f"  JSON: {WRONG_PREDICTIONS_JSON}")
    print(f"  CSV: {WRONG_PREDICTIONS_CSV}")
    print(f"  Summary: {WRONG_PREDICTIONS_SUMMARY_JSON}")





In [21]:
def evaluate_condition(
    model,
    tokenizer,
    eval_df: pd.DataFrame,
    condition_name: str,
    ablation_heads=None,
) -> dict:
    """Run evaluation on all rows in eval_df. Returns per-task accuracy dict."""
    results = {}
    total = len(eval_df)

    for i, (_, row) in enumerate(eval_df.iterrows(), start=1):
        if pd.isna(row["haystack_text"]) or pd.isna(row["needle_sentence"]):
            print(f"[{condition_name}] {i}/{total} SKIP - missing text data")
            continue

        task_id = row["task"]
        task = TASK_MAP[task_id]

        if task_id not in results:
            results[task_id] = {"attempts": 0, "matches": 0, "samples": []}

        sample_result = evaluate_sample(model, tokenizer, row, task, ablation_heads)

        if sample_result["skipped"]:
            continue

        results[task_id]["attempts"] += 1
        if sample_result["value_match"]:
            results[task_id]["matches"] += 1

        row_index = row.name if isinstance(row.name, (int, np.integer)) else None
        sample_with_context = dict(sample_result)
        sample_with_context["row_index"] = int(row_index) if row_index is not None else None
        sample_with_context["filename"] = row.get("filename") if hasattr(row, "get") else None
        results[task_id]["samples"].append(sample_with_context)

        if i % 10 == 0:
            print(f"[{condition_name}] {i}/{total}")

    # Compute accuracy per task
    for task_id in results:
        r = results[task_id]
        r["accuracy"] = r["matches"] / max(1, r["attempts"])

    return results

---
## Section 5 -- Experiment 1: Baseline Evaluation (No Ablation)

In [22]:
os.makedirs(ABLATION_OUTPUT_DIR, exist_ok=True)
if ENABLE_ERROR_LOGGING:
    os.makedirs(ERROR_LOG_DIR, exist_ok=True)

all_wrong_predictions = []

baseline_results = evaluate_condition(model, tokenizer, ablation_df, "baseline")
if ENABLE_ERROR_LOGGING:
    all_wrong_predictions.extend(
        collect_wrong_predictions(
            baseline_results,
            condition_name="baseline",
            experiment_type="baseline",
            heads_ablated=[],
        )
    )

print("\nBaseline results:")
for task_id, r in baseline_results.items():
    print(f"  {task_id:<35} {r['matches']}/{r['attempts']}  ({r['accuracy']:.1%})")

[baseline] 10/265


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SGDClassifier from version 1.8.0 when using version 1.6.1. This might lead to breaking code o

[baseline] 20/265
[baseline] 30/265
[baseline] 50/265
[baseline] 60/265
[baseline] 70/265
[baseline] 80/265
[baseline] 90/265
[baseline] 100/265
[baseline] 110/265
[baseline] 120/265
[baseline] 130/265
[baseline] 140/265
[baseline] 150/265
[baseline] 160/265
[baseline] 180/265
[baseline] 190/265
[baseline] 200/265
[baseline] 210/265
[baseline] 220/265
[baseline] 230/265
[baseline] 240/265
[baseline] 250/265
[baseline] 260/265

Baseline results:
  employees_count_total               28/28  (100.0%)
  incorporation_year                  27/31  (87.1%)
  registrant_name                     38/39  (97.4%)
  headquarters_city                   28/29  (96.6%)
  ceo_lastname                        24/31  (77.4%)
  headquarters_state                  26/27  (96.3%)
  incorporation_state                 30/31  (96.8%)
  holder_record_amount                23/29  (79.3%)
  employees_count_full_time           10/10  (100.0%)


In [23]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "baseline_results.json"), "w") as f:
    # Strip sample-level decoded text for cleaner JSON
    export = {tid: {"attempts": r["attempts"], "matches": r["matches"], "accuracy": r["accuracy"]} for tid, r in baseline_results.items()}
    json.dump(export, f, indent=2)
    print(f"Saved baseline results.")

Saved baseline results.


---
## Section 6 -- Experiment 2: Within-Task Ablation

For each task, ablate that task's own top-K heads at increasing K values.

In [24]:
within_task_results = {}

for task_id in TASK_MAP:
    task_df = ablation_df[ablation_df["task"] == task_id].reset_index(drop=True)
    if len(task_df) == 0:
        print(f"SKIP {task_id} -- no ablation samples")
        continue

    available_heads = task_head_rankings.get(task_id, [])
    if not available_heads:
        print(f"SKIP {task_id} -- no ranked heads")
        continue

    within_task_results[task_id] = {}

    for k in ABLATION_K_VALUES:
        heads_to_ablate = available_heads[:k]
        if not heads_to_ablate:
            break

        condition_name = f"within_{task_id}_k{k}"
        results = evaluate_condition(model, tokenizer, task_df, condition_name, ablation_heads=heads_to_ablate)
        if ENABLE_ERROR_LOGGING:
            all_wrong_predictions.extend(
                collect_wrong_predictions(
                    results,
                    condition_name=condition_name,
                    experiment_type="within_task",
                    heads_ablated=heads_to_ablate,
                )
            )

        r = results.get(task_id, {"attempts": 0, "matches": 0, "accuracy": 0.0})
        within_task_results[task_id][k] = {
            "k": k,
            "heads_ablated": len(heads_to_ablate),
            "attempts": r["attempts"],
            "matches": r["matches"],
            "accuracy": r["accuracy"],
        }

        print(f"  {task_id} k={k}: {r['matches']}/{r['attempts']}  ({r['accuracy']:.1%})")

[within_registrant_name_k8] 10/42
[within_registrant_name_k8] 20/42
[within_registrant_name_k8] 30/42
[within_registrant_name_k8] 40/42
  registrant_name k=8: 32/39  (82.1%)
[within_registrant_name_k16] 10/42
[within_registrant_name_k16] 20/42
[within_registrant_name_k16] 30/42
[within_registrant_name_k16] 40/42
  registrant_name k=16: 32/39  (82.1%)
[within_registrant_name_k32] 10/42
[within_registrant_name_k32] 20/42
[within_registrant_name_k32] 30/42
[within_registrant_name_k32] 40/42
  registrant_name k=32: 27/39  (69.2%)
[within_registrant_name_k48] 10/42
[within_registrant_name_k48] 20/42
[within_registrant_name_k48] 30/42
[within_registrant_name_k48] 40/42
  registrant_name k=48: 11/39  (28.2%)
[within_registrant_name_k64] 10/42
[within_registrant_name_k64] 20/42
[within_registrant_name_k64] 30/42
[within_registrant_name_k64] 40/42
  registrant_name k=64: 14/39  (35.9%)
[within_registrant_name_k96] 10/42
[within_registrant_name_k96] 20/42
[within_registrant_name_k96] 30/42
[with

In [25]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "within_task_ablation.json"), "w") as f:
    json.dump(within_task_results, f, indent=2)
    print("Saved within-task ablation results.")

Saved within-task ablation results.


---
## Section 7 -- Experiment 3: Across-Task Ablation (Shared Heads)

Ablate the global shared retrieval heads and measure accuracy across all tasks.

In [26]:
across_task_results = {}

for k in ABLATION_K_VALUES:
    heads_to_ablate = shared_heads[:k]
    if not heads_to_ablate:
        print(f"Only {len(shared_heads)} shared heads available, stopping at k={k}")
        break

    condition_name = f"across_task_k{k}"
    results = evaluate_condition(model, tokenizer, ablation_df, condition_name, ablation_heads=heads_to_ablate)
    if ENABLE_ERROR_LOGGING:
        all_wrong_predictions.extend(
            collect_wrong_predictions(
                results,
                condition_name=condition_name,
                experiment_type="across_task",
                heads_ablated=heads_to_ablate,
            )
        )

    total_attempts = sum(r["attempts"] for r in results.values())
    total_matches  = sum(r["matches"] for r in results.values())
    overall_acc    = total_matches / max(1, total_attempts)

    across_task_results[k] = {
        "k": k,
        "heads_ablated": len(heads_to_ablate),
        "overall_accuracy": overall_acc,
        "total_attempts": total_attempts,
        "total_matches": total_matches,
        "per_task": {tid: {"attempts": r["attempts"], "matches": r["matches"], "accuracy": r["accuracy"]} for tid, r in results.items()},
    }

    print(f"k={k}: {total_matches}/{total_attempts}  ({overall_acc:.1%})")

[across_task_k8] 10/265
[across_task_k8] 20/265
[across_task_k8] 30/265
[across_task_k8] 50/265
[across_task_k8] 60/265
[across_task_k8] 70/265
[across_task_k8] 80/265
[across_task_k8] 90/265
[across_task_k8] 100/265
[across_task_k8] 110/265
[across_task_k8] 120/265
[across_task_k8] 130/265
[across_task_k8] 140/265
[across_task_k8] 150/265
[across_task_k8] 160/265
[across_task_k8] 180/265
[across_task_k8] 190/265
[across_task_k8] 200/265
[across_task_k8] 210/265
[across_task_k8] 220/265
[across_task_k8] 230/265
[across_task_k8] 240/265
[across_task_k8] 250/265
[across_task_k8] 260/265
k=8: 229/255  (89.8%)
[across_task_k16] 10/265
[across_task_k16] 20/265
[across_task_k16] 30/265
[across_task_k16] 50/265
[across_task_k16] 60/265
[across_task_k16] 70/265
[across_task_k16] 80/265
[across_task_k16] 90/265
[across_task_k16] 100/265
[across_task_k16] 110/265
[across_task_k16] 120/265
[across_task_k16] 130/265
[across_task_k16] 140/265
[across_task_k16] 150/265
[across_task_k16] 160/265
[acr

In [27]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "across_task_ablation.json"), "w") as f:
    json.dump(across_task_results, f, indent=2)
    print("Saved across-task ablation results.")

Saved across-task ablation results.


---
## Section 7b -- Experiment 4: Global Heads Ablation
Ablate the global powerhouse retrieval heads and measure accuracy across all tasks.

In [28]:
global_task_results = {}

for k in ABLATION_K_VALUES:
    heads_to_ablate = global_heads[:k]
    if not heads_to_ablate:
        print(f"Only {len(global_heads)} global heads available, stopping at k={k}")
        break

    condition_name = f"global_task_k{k}"
    results = evaluate_condition(model, tokenizer, ablation_df, condition_name, ablation_heads=heads_to_ablate)
    if ENABLE_ERROR_LOGGING:
        all_wrong_predictions.extend(
            collect_wrong_predictions(
                results,
                condition_name=condition_name,
                experiment_type="global",
                heads_ablated=heads_to_ablate,
            )
        )

    total_attempts = sum(r["attempts"] for r in results.values())
    total_matches  = sum(r["matches"] for r in results.values())
    overall_acc    = total_matches / max(1, total_attempts)

    global_task_results[k] = {
        "k": k,
        "heads_ablated": len(heads_to_ablate),
        "overall_accuracy": overall_acc,
        "total_attempts": total_attempts,
        "total_matches": total_matches,
        "per_task": {tid: {"attempts": r["attempts"], "matches": r["matches"], "accuracy": r["accuracy"]} for tid, r in results.items()},
    }

    print(f"Global k={k}: {total_matches}/{total_attempts}  ({overall_acc:.1%})")

with open(os.path.join(ABLATION_OUTPUT_DIR, "global_ablation.json"), "w") as f:
    json.dump(global_task_results, f, indent=2)
    print("Saved global-task ablation results.")


[global_task_k8] 10/265
[global_task_k8] 20/265
[global_task_k8] 30/265
[global_task_k8] 50/265
[global_task_k8] 60/265
[global_task_k8] 70/265
[global_task_k8] 80/265
[global_task_k8] 90/265
[global_task_k8] 100/265
[global_task_k8] 110/265
[global_task_k8] 120/265
[global_task_k8] 130/265
[global_task_k8] 140/265
[global_task_k8] 150/265
[global_task_k8] 160/265
[global_task_k8] 180/265
[global_task_k8] 190/265
[global_task_k8] 200/265
[global_task_k8] 210/265
[global_task_k8] 220/265
[global_task_k8] 230/265
[global_task_k8] 240/265
[global_task_k8] 250/265
[global_task_k8] 260/265
Global k=8: 229/255  (89.8%)
[global_task_k16] 10/265
[global_task_k16] 20/265
[global_task_k16] 30/265
[global_task_k16] 50/265
[global_task_k16] 60/265
[global_task_k16] 70/265
[global_task_k16] 80/265
[global_task_k16] 90/265
[global_task_k16] 100/265
[global_task_k16] 110/265
[global_task_k16] 120/265
[global_task_k16] 130/265
[global_task_k16] 140/265
[global_task_k16] 150/265
[global_task_k16] 160/2

---
## Section 8 -- Random Head Ablation Control

Ablate randomly selected heads to prove the accuracy drop is specific to retrieval heads.

In [29]:
num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads

# All possible (layer, head) pairs, excluding the shared retrieval heads
shared_set = set(shared_heads)
all_heads = [(l, h) for l in range(num_layers) for h in range(num_heads) if (l, h) not in shared_set]

random.seed(SPLIT_SEED)
random_pool = random.sample(all_heads, min(max(ABLATION_K_VALUES), len(all_heads)))

random_ablation_results = {}

for k in ABLATION_K_VALUES:
    heads_to_ablate = random_pool[:k]

    condition_name = f"random_k{k}"
    results = evaluate_condition(model, tokenizer, ablation_df, condition_name, ablation_heads=heads_to_ablate)
    if ENABLE_ERROR_LOGGING:
        all_wrong_predictions.extend(
            collect_wrong_predictions(
                results,
                condition_name=condition_name,
                experiment_type="random",
                heads_ablated=heads_to_ablate,
            )
        )

    total_attempts = sum(r["attempts"] for r in results.values())
    total_matches  = sum(r["matches"] for r in results.values())
    overall_acc    = total_matches / max(1, total_attempts)

    random_ablation_results[k] = {
        "k": k,
        "heads_ablated": [(l, h) for l, h in heads_to_ablate],
        "overall_accuracy": overall_acc,
        "total_attempts": total_attempts,
        "total_matches": total_matches,
        "per_task": {tid: {"attempts": r["attempts"], "matches": r["matches"], "accuracy": r["accuracy"]} for tid, r in results.items()},
    }

    print(f"random k={k}: {total_matches}/{total_attempts}  ({overall_acc:.1%})")

[random_k8] 10/265
[random_k8] 20/265
[random_k8] 30/265
[random_k8] 50/265
[random_k8] 60/265
[random_k8] 70/265
[random_k8] 80/265
[random_k8] 90/265
[random_k8] 100/265
[random_k8] 110/265
[random_k8] 120/265
[random_k8] 130/265
[random_k8] 140/265
[random_k8] 150/265
[random_k8] 160/265
[random_k8] 180/265
[random_k8] 190/265
[random_k8] 200/265
[random_k8] 210/265
[random_k8] 220/265
[random_k8] 230/265
[random_k8] 240/265
[random_k8] 250/265
[random_k8] 260/265
random k=8: 237/255  (92.9%)
[random_k16] 10/265
[random_k16] 20/265
[random_k16] 30/265
[random_k16] 50/265
[random_k16] 60/265
[random_k16] 70/265
[random_k16] 80/265
[random_k16] 90/265
[random_k16] 100/265
[random_k16] 110/265
[random_k16] 120/265
[random_k16] 130/265
[random_k16] 140/265
[random_k16] 150/265
[random_k16] 160/265
[random_k16] 180/265
[random_k16] 190/265
[random_k16] 200/265
[random_k16] 210/265
[random_k16] 220/265
[random_k16] 230/265
[random_k16] 240/265
[random_k16] 250/265
[random_k16] 260/265
ran

In [30]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "random_ablation_control.json"), "w") as f:
    json.dump(random_ablation_results, f, indent=2)
    print("Saved random ablation control results.")

Saved random ablation control results.


---
## Section 9 -- Experiment 5: Cross-Task Within-Heads Transfer

For each source task, ablate its top-K within-task heads while evaluating every target task.
Baseline metrics are used as the K=0 reference (no K=0 ablation run).

In [31]:
CROSS_TASK_RESULTS_JSON = os.path.join(ABLATION_OUTPUT_DIR, "cross_task_within_heads_ablation.json")
cross_task_within_heads_results = {}

for k in ABLATION_K_VALUES:
    k_key = str(k)
    cross_task_within_heads_results[k_key] = {
        "k": k,
        "per_source_task": {},
    }

    for source_task_id in TASK_MAP.keys():
        source_head_ranking = task_head_rankings.get(source_task_id, [])
        heads_to_ablate = source_head_ranking[:k]

        if not heads_to_ablate:
            continue

        source_results = {}

        for target_task_id in TASK_MAP.keys():
            target_df = ablation_df[ablation_df["task"] == target_task_id].reset_index(drop=True)

            if len(target_df) == 0:
                source_results[target_task_id] = {
                    "attempts": 0,
                    "matches": 0,
                    "accuracy": 0.0,
                    "heads_ablated": len(heads_to_ablate),
                }
                continue

            condition_name = f"cross_src_{source_task_id}_tgt_{target_task_id}_k{k}"
            condition_results = evaluate_condition(
                model,
                tokenizer,
                target_df,
                condition_name,
                ablation_heads=heads_to_ablate,
            )

            if ENABLE_ERROR_LOGGING:
                all_wrong_predictions.extend(
                    collect_wrong_predictions(
                        condition_results,
                        condition_name=condition_name,
                        experiment_type="cross_task_within",
                        heads_ablated=heads_to_ablate,
                    )
                )

            target_metrics = condition_results.get(target_task_id, {"attempts": 0, "matches": 0, "accuracy": 0.0})
            source_results[target_task_id] = {
                "attempts": target_metrics.get("attempts", 0),
                "matches": target_metrics.get("matches", 0),
                "accuracy": target_metrics.get("accuracy", 0.0),
                "heads_ablated": len(heads_to_ablate),
            }

        cross_task_within_heads_results[k_key]["per_source_task"][source_task_id] = source_results
        print(f"cross-task source={source_task_id} k={k} complete")

with open(CROSS_TASK_RESULTS_JSON, "w") as f:
    json.dump(cross_task_within_heads_results, f, indent=2)

print(f"Saved cross-task within-heads results: {CROSS_TASK_RESULTS_JSON}")

[cross_src_registrant_name_tgt_registrant_name_k8] 10/42
[cross_src_registrant_name_tgt_registrant_name_k8] 20/42
[cross_src_registrant_name_tgt_registrant_name_k8] 30/42
[cross_src_registrant_name_tgt_registrant_name_k8] 40/42
[cross_src_registrant_name_tgt_headquarters_city_k8] 10/29
[cross_src_registrant_name_tgt_headquarters_city_k8] 20/29
[cross_src_registrant_name_tgt_headquarters_state_k8] 10/28
[cross_src_registrant_name_tgt_headquarters_state_k8] 20/28
[cross_src_registrant_name_tgt_incorporation_state_k8] 10/32
[cross_src_registrant_name_tgt_incorporation_state_k8] 20/32
[cross_src_registrant_name_tgt_incorporation_state_k8] 30/32
[cross_src_registrant_name_tgt_incorporation_year_k8] 10/32
[cross_src_registrant_name_tgt_incorporation_year_k8] 20/32
[cross_src_registrant_name_tgt_incorporation_year_k8] 30/32
[cross_src_registrant_name_tgt_employees_count_total_k8] 10/29
[cross_src_registrant_name_tgt_employees_count_total_k8] 20/29
[cross_src_registrant_name_tgt_holder_record_

In [40]:
os.makedirs(ABLATION_OUTPUT_DIR, exist_ok=True)
if ENABLE_ERROR_LOGGING:
    os.makedirs(ERROR_LOG_DIR, exist_ok=True)

In [32]:
meta = {
    "experiment_name": EXPERIMENT_NAME,
    "experiment_run_name": ABLATION_RUN_NAME,
    "source_ranking": RANKING_RUN_NAME,
    "timestamp": TIMESTAMP,
    "ablation_k_values": ABLATION_K_VALUES,
}

# If SOURCE_MODEL exists also include it in meta
if SOURCE_MODEL:
    meta["source_model"] = SOURCE_MODEL

with open(os.path.join(ABLATION_OUTPUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Saved meta.json")

if ENABLE_ERROR_LOGGING:
    write_wrong_prediction_logs(all_wrong_predictions)
    print(f"Total wrong rows logged: {len(all_wrong_predictions)}")

Saved meta.json
Saved wrong prediction logs to: /content/ablation_2026-03-22_20-46-39/error_logs
  JSON: /content/ablation_2026-03-22_20-46-39/error_logs/wrong_predictions.json
  CSV: /content/ablation_2026-03-22_20-46-39/error_logs/wrong_predictions.csv
  Summary: /content/ablation_2026-03-22_20-46-39/error_logs/wrong_predictions_summary.json
Total wrong rows logged: 11319


In [45]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "baseline_results.json"), "w") as f:
    # Strip sample-level decoded text for cleaner JSON
    export = {tid: {"attempts": r["attempts"], "matches": r["matches"], "accuracy": r["accuracy"]} for tid, r in baseline_results.items()}
    json.dump(export, f, indent=2)
    print(f"Saved baseline results.")

Saved baseline results.


In [46]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "within_task_ablation.json"), "w") as f:
    json.dump(within_task_results, f, indent=2)
    print("Saved within-task ablation results.")

Saved within-task ablation results.


In [47]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "across_task_ablation.json"), "w") as f:
    json.dump(across_task_results, f, indent=2)
    print("Saved across-task ablation results.")

Saved across-task ablation results.


In [48]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "random_ablation_control.json"), "w") as f:
    json.dump(random_ablation_results, f, indent=2)
    print("Saved random ablation control results.")

Saved random ablation control results.


In [49]:
with open(os.path.join(ABLATION_OUTPUT_DIR, "global_ablation.json"), "w") as f:
    json.dump(global_task_results, f, indent=2)
    print("Saved global-task ablation results.")

Saved global-task ablation results.


In [33]:
import shutil
from google.colab import files

# Define the folder path and the output zip file name
folder_path = ABLATION_OUTPUT_DIR
output_filename = f"{ABLATION_OUTPUT_DIR}.zip"

# Create a zip file of the folder
shutil.make_archive(output_filename.replace('.zip', ''), 'zip', folder_path)

print(f"Zipped folder to {output_filename}")

# Trigger the download
files.download(output_filename)

Zipped folder to /content/ablation_2026-03-22_20-46-39.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>